## **CMIP Climate Stripes with Julia**

#### **Purpose of this Tutorial**
* In this Script we will analyse the Climate Data using the Julia programming language. 
* Juia is an open source high level programming language (like Python, Matlab, R,...) that was published in 2012 and is mainly used for scientific programming. 
* The main difference of Julia compared to Python, R or Matlab is its speed. We will explore this characteristic in the following tutorial generating the climate stripes and comparing the performance to a Python implementation

#### **Setup**
1. Download Julia from https://github.com/JuliaLang/juliaup#installation
2. Install Julia extension in Visual Code Studio market place
3. Code can be execute in 
    - Julia Read-Eval-Print-Loop (Julia: Start REPL)
    - Jupyter Notebook
    - ...
4. When selecting the Kernel in Jupyter:
    - Julia release channel (automatically uses most recent version of Julia on your machine)

In [1]:
print("Hello World")

Hello World

#### **Package management**
- For different projects you often need different additional packages or libraries
- In Julia the `pkg` helps you to manage libraries and make your project reproducible for others
    - You have a global environment in `~/.julia/environments/v1.13/` (in case of version 1.13)
    - You should create a local environment in your project folder
    - All packages are installed into the same directory `~/.julia/packages/`, independent of the environment
- The environent contains two files
    - `Project.toml`
    - `Manifest.toml`

In [2]:
using Pkg # tell Julia to use pkg manager
Pkg.activate() # activate the global environment (should already be the case)
Base.active_project() # show which Project.toml file is currently active

print(Base.load_path()) # shows all paths to environments that are currently checked when loading packages
print("\n") # print a linebreak
print(LOAD_PATH) # print LOAD_PATH variable, showing all environments which will currently be checked

  Activating project at `~/.julia/environments/v1.13`


["/Users/tim/.julia/environments/v1.13/Project.toml", "/Users/tim/.julia/juliaup/julia-1.13.1+0.aarch64.apple.darwin14/Julia-1.13.app/Contents/Resources/julia/share/julia/stdlib/v1.13"]
["@", "@v#.#", "@stdlib"]

In [3]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to global environment
using BenchmarkTools # call BenchmarkTools from global environment

     Project No packages added to or removed from `~/.julia/environments/v1.13/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.13/Manifest.toml`


In [4]:
Pkg.activate(".") # activate current directory as project environment
using BenchmarkTools # call BenchmarkTools (global environment will be used)

  Activating new project at `~/Documents/GitHub/Julia-CMIP`


In [5]:
deleteat!(LOAD_PATH, 2) # remove the global environment from LOAD_PATH so that BenchmarkTools is not found when using it in this notebook
using BenchmarkTools # this should now throw an error since BenchmarkTools is not installed in the current environment

ArgumentError: ArgumentError: Package BenchmarkTools not found in current path, maybe you meant `import/using .BenchmarkTools`.
- Otherwise, run `import Pkg; Pkg.add("BenchmarkTools")` to install the BenchmarkTools package.

In [6]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to local environment (create Project.toml and Manifest.toml files)
using BenchmarkTools

   Resolving package versions...
    Updating `~/Documents/GitHub/Julia-CMIP/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
    Updating `~/Documents/GitHub/Julia-CMIP/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
  [34da2185] + Compat v4.18.1
  [682c06a0] + JSON v1.10.0
  [69de0a69] + Parsers v3.0.0
  [aea7be01] + PrecompileTools v1.3.4
  [21216c6a] + Preferences v1.6.0
  [10745b16] + Statistics v1.11.5
  [ec057cc2] + StructUtils v2.9.2
  [56f22d72] + Artifacts v1.11.0
  [ade2ca70] + Dates v1.11.0
  [8f399da3] + Libdl v1.11.0
  [37e2e46d] + LinearAlgebra v1.13.0
  [56ddb016] + Logging v1.11.0
  [de0858da] + Printf v1.11.0
  [9abbd945] + Profile v1.11.0
  [9a3f8284] + Random v1.11.0
  [ea8e919c] + SHA v1.0.0
  [f489334b] + StyledStrings v1.11.0
  [fa267f1f] + TOML v1.0.3
  [cf7118a7] + UUIDs v1.11.0
  [4ec0a83e] + Unicode v1.11.0
  [e66e0078] + CompilerSupportLibraries_jll v1.5.5+2
  [4536629a] + OpenBLAS_jll v0.3.30+0
  [8e850b90] + libblastrampoline_jll v5.15.0+0


#### **CMIP Data Analysis: Packages used**
* **Rasters**: Julia package for labeled multidimensional array and geospatial raster manipulation (Python equivalent: `xarray` / `rioxarray`).
* **NCDatasets**: Backend library for reading, writing, and inspecting NetCDF files (Python equivalent: `netCDF4`).
* **Dates**: Built-in standard library for handling calendar dates, timestamps, and time intervals (Python equivalent: `datetime`).
* **Statistics**: Built-in standard library providing basic statistical reductions like `mean` and `std` (Python equivalent: `numpy` statistical functions).
* **CairoMakie**: High-performance, publication-quality 2D vector and static plotting backend for the Makie ecosystem (Python equivalent: `matplotlib`).
* **GeoMakie**: Cartographic extension for Makie that adds map projections and coastlines (Python equivalent: `cartopy`).
* **BenchmarkTools**: Precision performance tracking library that handles JIT compilation warm-up and statistical timing (Python equivalent: ´timeit`).
* **PythonCall**: Package that allows to call Python from Julia (loads numpy library as specified in `CondaPkg.toml`)

In [1]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.add(["Rasters", "NCDatasets", "Dates", "Statistics", "CairoMakie", "GeoMakie", "PythonCall", "BenchmarkTools"])
# Pkg.instantiate()

using Rasters, NCDatasets, Dates, Statistics
using CairoMakie, GeoMakie
using PythonCall, BenchmarkTools

  Activating project at `~/Documents/GitHub/Julia-CMIP`
   Resolving package versions...
    Updating `~/Documents/GitHub/Julia-CMIP/Project.toml`
  [13f3f980] + CairoMakie v0.15.15
  [db073c08] + GeoMakie v0.7.17
  [85f8d34a] + NCDatasets v0.14.15
  [6099a3de] + PythonCall v0.9.36
  [a3a2b9e3] + Rasters v0.15.0
  [10745b16] + Statistics v1.11.5
  [ade2ca70] + Dates v1.11.0
    Updating `~/Documents/GitHub/Julia-CMIP/Manifest.toml`
  [621f4979] + AbstractFFTs v1.5.0
  [1520ce14] + AbstractTrees v0.4.5
  [7d9f7c33] + Accessors v0.1.45
  [79e6a3ab] + Adapt v4.7.4
  [35492f91] + AdaptivePredicates v1.2.0
  [66dad0bd] + AliasTables v1.1.3
  [27a7e980] + Animations v0.4.2
  [67c07d97] + Automa v1.2.0
  [13072b0f] + AxisAlgorithms v1.1.0
  [39de3d68] + AxisArrays v0.4.8
  [18cc8868] + BaseDirs v1.4.0
  [fa961155] + CEnum v0.5.0
  [179af706] + CFTime v0.2.10
  [96374032] + CRlibm v1.0.2
  [159f3aea] + Cairo v1.1.1
  [13f3f980] + CairoMakie v0.15.15
  [d360d2e6] + ChainRulesCore v1.26.1
  [6b3

#### **Python vs Julia**
**Syntax**
- Julia has 1-based indexing (vs. Python 0 based), e.g. `x=[0,1,2], x[1]=0`
- Julia needs `end` keyword for block termination but does not care about indentation 
- String vs Character: Julia differentiates between characters (single quotes: `'h'`) and strings (doube quotes: `"hello world"`)

**Design**
- Both Python and Julia are high level languages (human readible)
- Both languages use dynamics typing, so variables types do not need to be defiend beforehand
- However, they differ in the way how code is translated to machine code
    - **Python** is an interpreter languge and compiles code into intermediate bytecode (saved in `__pycache__`), which the interpreter then executes step-by-step at runtime without compiling it into native CPU machine code.
    - **Julia** uses JIT (just in time) compilation, where a function is compiled just before it is called based on the type of the arguments that are passed to the function (multiple dispatch)
    - In **C** the whole program code is compiled before execution

**Speed**
- Python has functions that are precompiled in **C**
    - If you use only these functions, python will not be slower than Julia
    - If you use a lot of custom functions with loops and conditionals, they will be faster in Julia
- Generally
    - Use python for deep learning, tabular data processing (pandas) or general software engineering (object oriented)
    - Use julia for scientific coding (differential equations, ABMs, physical models, ...) and mathematics and optimisation

In the following we are going to showcase these performance differences between Python and Julia based on the CMIP data. 

In [2]:
# Use PythonCall to import Pythons xarray and numpy
xr = pyimport("xarray")
np = pyimport("numpy");

In [5]:
path = "/Users/tim/Documents/GitHub/Julia-CMIP/"
file = "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc"
filepath = joinpath(path, file)

# Load in Julia and convert to concrete Float32 (since tas has no missing values)
Rasters.checkmem!(false) # Turn off the macOS free-RAM check globally for the session
tas_raw_jl = Raster(filepath, name=:tas)

┌ 192×144×1980 Raster{Union{Missing, Float32}, 3} tas ┐
├─────────────────────────────────────────────────────┴────────────────── dims ┐
  ↓ X Mapped{Float64} 0.9375:1.875:359.0625 ForwardOrdered Regular Intervals{Center},
  → Y Mapped{Float64} -89.375:1.25:89.375 ForwardOrdered Regular Intervals{Center},
  ↗ Ti Sampled{DateTime} [DateTime("1850-01-16T12:00:00"), …, DateTime("2014-12-16T12:00:00")] ForwardOrdered Explicit Intervals{Center}
├──────────────────────────────────────────────────────────────────── metadata ┤
  Metadata{Rasters.NCDsource} of Dict{String, Any} with 10 entries:
  "cell_methods"  => "area: time: mean"
  "coordinates"   => "height"
  "_FillValue"    => 1.0f20
  "missing_value" => 1.0f20
  "standard_name" => "air_temperature"
  "long_name"     => "Near-Surface Air Temperature"
  "units"         => "K"
  "history"       => "2019-11-08T06:41:45Z altered by CMOR: Treated scalar dime…
  "comment"       => "near-surface (usually, 2 meter) air temperature"
  "cell_measu

**Raster File Summary**
- First Section (`192×144×1980 Raster{Union{Missing, Float32}, 3} tas`): three dimensions, each value can either be float or Missing
    - The file acutally does not contain missing values
    - Doing the calculation with one variable type increases speed dramatically
- Second Section: Axis dimensions, longitude, latitude and time stamp
- Third Section: Metadata
- Fourth Section: Spatial and temporal extend and coordinate system

In [7]:
tas_clean_jl = modify(Array{Float32}, tas_raw_jl);

In [8]:
# Load in Python via xarray (.load() forces it into RAM for a fair benchmark)
ds_py = xr.open_dataset(filepath).load()
tas_raw_py = ds_py["tas"]

Python:
<xarray.DataArray 'tas' (time: 1980, lat: 144, lon: 192)> Size: 219MB
array([[[252.68433, 252.65332, 252.6128 , ..., 252.87817, 252.87573,
         252.87354],
        [253.29565, 253.28394, 253.27759, ..., 253.29736, 253.2898 ,
         253.22339],
        [253.3252 , 253.30762, 253.29004, ..., 253.55713, 253.46411,
         253.42651],
        ...,
        [237.47144, 237.4419 , 237.44238, ..., 237.82227, 237.68164,
         237.52588],
     ... 36 more lines ...
    height   float64 8B 2.0
Attributes:
    standard_name:  air_temperature
    long_name:      Near-Surface Air Temperature
    comment:        near-surface (usually, 2 meter) air temperature
    units:          K
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    history:        2019-11-08T06:41:45Z altered by CMOR: Treated scalar dime...

In [27]:
function calc_anom_native_jl(tas_raw)
    # 1. Convert to °C
    tas = tas_raw .- 273.15f0
    
    # 2. Slice reference (360 months) and analysis (432 months) periods
    tas_ref    = @view tas[Ti(DateTime(1981, 1, 1) .. DateTime(2011, 1, 1))]
    tas_period = tas[Ti(DateTime(1979, 1, 1) .. DateTime(2015, 1, 1))]
    
    nx, ny, nt_ref = size(tas_ref)
    nt_per = size(tas_period, 3)
    
    # 3. Compute 12 monthly climatologies (192 × 144 × 12)
    clim = zeros(Float32, nx, ny, 12)
    n_years_ref = nt_ref ÷ 12
    
    @inbounds for m in 1:12
        for k in 0:(n_years_ref - 1)
            t = m + k * 12
            @simd for i in 1:(nx * ny)
                clim[i + (m - 1) * nx * ny] += tas_ref.data[i + (t - 1) * nx * ny]
            end
        end
    end
    clim ./= n_years_ref
    
    # 4. Subtract monthly climatology in-place
    tas_anom = copy(tas_period)
    @inbounds for t in 1:nt_per
        m = mod1(t, 12)
        for y in 1:ny
            @simd for x in 1:nx
                tas_anom.data[x, y, t] -= clim[x, y, m]
            end
        end
    end
    
    return tas_anom
end

calc_anom_native_jl (generic function with 1 method)

In [28]:
@pyexec """
def calc_anom_native_py(tas_da):
    import numpy as np  # <-- Put the import INSIDE the function!
    
    # Slice periods using xarray, then extract raw 3D numpy arrays (time, lat, lon)
    ref_arr = (tas_da.sel(time=slice('1981-01-01', '2010-12-31')).values - 273.15)
    per_arr = (tas_da.sel(time=slice('1979-01-01', '2014-12-31')).values - 273.15)
    
    nt_ref, ny, nx = ref_arr.shape
    nt_per = per_arr.shape[0]
    
    # 1. Compute 12 monthly climatologies using native Python loops
    clim = np.zeros((12, ny, nx), dtype=np.float32)
    for m in range(12):
        for y in range(ny):
            for x in range(nx):
                total = 0.0
                count = 0
                for t in range(m, nt_ref, 12):
                    total += ref_arr[t, y, x]
                    count += 1
                clim[m, y, x] = total / count
                
    # 2. Subtract monthly climatology from 1979-2014 period
    anom = np.empty_like(per_arr)
    for t in range(nt_per):
        m = t % 12
        for y in range(ny):
            for x in range(nx):
                anom[t, y, x] = per_arr[t, y, x] - clim[m, y, x]
                
    return anom
""" => calc_anom_native_py

@pyexec """
def calc_anom_xarray_py(tas_da):
    tas = tas_da - 273.15
    tas_ref = tas.sel(time=slice('1981-01-01', '2010-12-31'))
    tas_clim = tas_ref.groupby('time.month').mean('time')
    
    # Optimization: slice 1979-2014 BEFORE groupby subtraction so we don't 
    # waste time computing anomalies for 1850-1978
    tas_period = tas.sel(time=slice('1979-01-01', '2014-12-31'))
    tas_anom = tas_period.groupby('time.month') - tas_clim
    return tas_anom.values
""" => calc_anom_xarray_py

Python: <function calc_anom_xarray_py at 0x14f2b6de0>

In [29]:
# ==============================================================================
# RUN BENCHMARKS
# ==============================================================================
println("1. Native Python (Interpreted for-loops):")
# We use @time (1 run) instead of @btime here because Python loops take ~5-8 seconds per run!
@time res_py_loop = calc_anom_native_py(tas_raw_py);

println("\n2. Python Fast Operation (Vectorized xarray groupby):")
@btime calc_anom_xarray_py($tas_raw_py);

println("\n3. Native Julia (JIT-compiled for-loops):")
@btime calc_anom_native_jl($tas_clean_jl);

# ==============================================================================
# VERIFY NUMERICAL ACCURACY
# ==============================================================================
res_xr_np = pyconvert(Array{Float32, 3}, calc_anom_xarray_py(tas_raw_py))
res_py_np = pyconvert(Array{Float32, 3}, res_py_loop)
res_jl    = Array(calc_anom_native_jl(tas_clean_jl))

# Note: Python arrays are (time, lat, lon) = (432, 144, 192), 
# while Julia Rasters are (lon, lat, time) = (192, 144, 432).
# We permute Python's dimensions to (3, 2, 1) to compare grid cell by grid cell:
res_xr_permuted = permutedims(res_xr_np, (3, 2, 1))
res_py_permuted = permutedims(res_py_np, (3, 2, 1))

println("\nAre all three results approximately equal (≈)? ", res_jl ≈ res_xr_permuted ≈ res_py_permuted)
println("Max absolute difference (Xarray vs Julia): ", maximum(abs.(res_jl .- res_xr_permuted)))

1. Native Python (Interpreted for-loops):
  4.105196 seconds (56 allocations: 2.203 KiB)

2. Python Fast Operation (Vectorized xarray groupby):
  29.690 ms (9 allocations: 208 bytes)

3. Native Julia (JIT-compiled for-loops):
  16.617 ms (21 allocations: 301.24 MiB)

Are all three results approximately equal (≈)? true
Max absolute difference (Xarray vs Julia): 0.0
